In [8]:
import cv2
import numpy as np

# Cargar imagen
imagen = cv2.imread("figuras.png")

# Convertir a escala de grises
gris = cv2.cvtColor(imagen, cv2.COLOR_BGR2GRAY)

# Umbral binario
_, umbral = cv2.threshold(gris, 127, 255, cv2.THRESH_BINARY_INV)

# Buscar contornos
contornos, _ = cv2.findContours(
    umbral,
    cv2.RETR_EXTERNAL,
    cv2.CHAIN_APPROX_SIMPLE
)

for contorno in contornos:

    area = cv2.contourArea(contorno)

    # Ignorar ruido
    if area < 100:
        continue

    # Calcular perímetro
    perimetro = cv2.arcLength(contorno, True)

    # Aproximar contorno
    aproximacion = cv2.approxPolyDP(
        contorno,
        0.04 * perimetro,
        True
    )

    vertices = len(aproximacion)

    # Calcular momentos de Hu
    momentos = cv2.moments(contorno)
    hu = cv2.HuMoments(momentos)

    # Clasificación
    if vertices == 3:
        figura = "Triangulo"

    elif vertices == 4:
        x, y, w, h = cv2.boundingRect(aproximacion)
        relacion = w / float(h)

        if 0.95 <= relacion <= 1.05:
            figura = "Cuadrado"
        else:
            figura = "Rectangulo"

    else:
        figura = "Circulo"

    # Centro de la figura
    M = cv2.moments(contorno)

    if M["m00"] != 0:
        cx = int(M["m10"] / M["m00"])
        cy = int(M["m01"] / M["m00"])
    else:
        cx, cy = 0, 0

    # Dibujar contorno
    cv2.drawContours(imagen, [contorno], -1, (0, 255, 0), 2)

    # Mostrar clasificación
    cv2.putText(
        imagen,
        figura,
        (cx - 40, cy),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.6,
        (255, 0, 0),
        2
    )

    print(f"\nFigura: {figura}")
    print("Momentos de Hu:")
    print(hu.flatten())

# Mostrar resultado
cv2.imshow("Clasificacion de Figuras", imagen)
cv2.waitKey(0)
cv2.destroyAllWindows()


Figura: Rectangulo
Momentos de Hu:
[0.34896422 0.09399825 0.         0.         0.         0.
 0.        ]

Figura: Triangulo
Momentos de Hu:
[ 1.92542532e-01  8.13185728e-06  4.57105902e-03  2.21792601e-07
  7.01555192e-12  6.29961582e-10 -8.08807521e-13]

Figura: Circulo
Momentos de Hu:
[ 1.59738116e-01  1.76558441e-04  1.26204019e-09  9.03204578e-12
 -9.59913342e-22 -1.13275417e-13  9.19585493e-23]
